In [8]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.neighbors import NearestNeighbors

In [12]:
df = pd.read_csv("US_Accidents_March23.csv",
                 usecols=["ID", "City", "State", "Start_Lat", "Start_Lng"])

In [13]:
df = df.dropna(subset=["City", "State"])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7728141 entries, 0 to 7728393
Data columns (total 5 columns):
 #   Column     Dtype  
---  ------     -----  
 0   ID         object 
 1   Start_Lat  float64
 2   Start_Lng  float64
 3   City       object 
 4   State      object 
dtypes: float64(2), object(3)
memory usage: 353.8+ MB


In [14]:
city = (df.groupby(["State", "City"])
          .agg(accidents=("ID", "count"))
          .reset_index())
city.info()

TOP_N = 15
city_top = (city.sort_values("accidents", ascending=False)
                .groupby("State").head(TOP_N))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20323 entries, 0 to 20322
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   State      20323 non-null  object
 1   City       20323 non-null  object
 2   accidents  20323 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 476.4+ KB


In [16]:

state = city.groupby("State", as_index=False)["accidents"].sum()

In [17]:
nodes = pd.concat([
    pd.DataFrame({"id": state.State, "label": state.State, "type": "state",
                  "state": state.State, "accidents": state.accidents}),
    pd.DataFrame({"id": city_top.State + "|" + city_top.City, "label": city_top.City,
                  "type": "city", "state": city_top.State,
                  "accidents": city_top.accidents}),
], ignore_index=True)
nodes.info()
nodes.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 781 entries, 0 to 780
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         781 non-null    object
 1   label      781 non-null    object
 2   type       781 non-null    object
 3   state      781 non-null    object
 4   accidents  781 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 30.6+ KB


,id,label,type,state,accidents
0,AL,AL,state,AL,101044
1,AR,AR,state,AR,22780
2,AZ,AZ,state,AZ,170609
3,CA,CA,state,CA,1741422
4,CO,CO,state,CO,90877


In [ ]:
# ---------- edges 1: state -> city ("contains") ----------
contains = pd.DataFrame({"source": city_top.State,
                         "target": city_top.State + "|" + city_top.City,
                         "type": "contains", "weight": city_top.accidents})
contains.info()
contains.head()

<class 'pandas.core.frame.DataFrame'>
Index: 732 entries, 3300 to 2900
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   source  732 non-null    object
 1   target  732 non-null    object
 2   type    732 non-null    object
 3   weight  732 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 28.6+ KB


,source,target,type,weight
3300,FL,FL|Miami,contains,186768
17483,TX,TX|Houston,contains,169428
1689,CA,CA|Los Angeles,contains,156491
10153,NC,NC|Charlotte,contains,138345
17308,TX,TX|Dallas,contains,130303


In [20]:
# ---------- edges 2: state <-> state ("border") from centroids ----------
c = df.dropna(subset=["Start_Lat", "Start_Lng", "State"])
cent = c.groupby("State")[["Start_Lat", "Start_Lng"]].median()
states = cent.index.to_numpy()

In [30]:
lat0 = np.deg2rad(cent.Start_Lat.mean())
xy = np.column_stack([cent.Start_Lng * np.cos(lat0), cent.Start_Lat])

K = 7
_, idx = NearestNeighbors(n_neighbors=K + 1).fit(xy).kneighbors(xy)

counts = {}
for i, js in enumerate(idx):
    for j in js[1:]:
        key = tuple(sorted((states[i], states[j])))
        counts[key] = counts.get(key, 0) + 1


print(idx)

[[ 0  9 23 40 16 38  1 15]
 [ 1 23 22 34 16 40 14  0]
 [ 2 30  3 42  4 31 48 11]
 [ 3  2 31 42 30 11 35 48]
 [ 4 48 30 42 27 34 39 14]
 [ 5 32 37 29 17 28 44  7]
 [ 6 18 43  7 36 29 32 47]
 [ 7 36 18  6 29 32 43  5]
 [ 8  9 38  0 25 16 23 40]
 [ 9  0 38 40 25 15 23 47]
 [10 27 14 22 39 21 12 46]
 [11 24 42 31 35 45 48  3]
 [12 46 13 20 15 33 10 21]
 [13 15 33 12 20 46 40 47]
 [14 22 27 10 34  1 39 12]
 [15 13 40 33 47 12  9 25]
 [16 23  0 41  1  9 40 34]
 [17 37 28  5 19 44 32 29]
 [18  6 36  7 43 29 32 47]
 [19 28 17 44 37  5 32 29]
 [20 33 12 46 13 47 15 36]
 [21 39 10 27 46 26 12 14]
 [22 14 10 27  1 34 12 39]
 [23 16  0  1 40  9 41 34]
 [24 11 42 48 45 35  4 31]
 [25 38 47 43  9  6 15 40]
 [26 39 21 27 10 48 14  4]
 [27 10 39 14 22 21 34 26]
 [28 17 37 44 19  5 32 29]
 [29 32  5  7 36 18 37  6]
 [30  2  4 48 42 34 41  3]
 [31 11  3 35 42 45  2 24]
 [32 29  5  7 36 37 17 28]
 [33 47 13 15 20 12 40  6]
 [34 14  1 22 41 27 23 10]
 [35 45 11 31 24 42  3 48]
 [36  7 18  6 29 32 43  5]
 

In [31]:

pairs = {k for k, n in counts.items() if n == 2}          # mutual links
# rescue states left with no link (e.g., FL)
_, idx3 = NearestNeighbors(n_neighbors=3).fit(xy).kneighbors(xy)
linked = {s for p in pairs for s in p}
for i, s in enumerate(states):
    if s not in linked:
        for j in idx3[i][1:]:
            pairs.add(tuple(sorted((s, states[j]))))

border = pd.DataFrame([(a, b, "border", 1) for a, b in sorted(pairs)],
                      columns=["source", "target", "type", "weight"])

In [32]:
edges = pd.concat([contains, border], ignore_index=True)

In [33]:

nodes.to_csv("nodes.csv", index=False)
edges.to_csv("edges.csv", index=False)
cent.to_csv("state_centroids.csv")
print(len(nodes), "nodes |", len(contains), "contains edges |", len(border), "border edges")

781 nodes | 732 contains edges | 141 border edges


In [39]:
print(edges[edges["type"]=="border"].to_string(index=False))

source target   type  weight
    AL     AR border       1
    AL     GA border       1
    AL     LA border       1
    AL     MS border       1
    AL     SC border       1
    AL     TN border       1
    AR     KS border       1
    AR     LA border       1
    AR     MO border       1
    AR     MS border       1
    AR     OK border       1
    AR     TN border       1
    AZ     CA border       1
    AZ     NM border       1
    AZ     NV border       1
    AZ     UT border       1
    CA     ID border       1
    CA     NM border       1
    CA     NV border       1
    CA     OR border       1
    CO     NM border       1
    CO     UT border       1
    CO     WY border       1
    CT     DE border       1
    CT     MA border       1
    CT     NH border       1
    CT     NJ border       1
    CT     NY border       1
    CT     RI border       1
    CT     VT border       1
    DC     DE border       1
    DC     MD border       1
    DC     NJ border       1
    DC     PA 